In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 21.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
from ultralytics import YOLO

print("✅ Drive đã mount!")

Mounted at /content/drive
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Drive đã mount!


In [3]:
# ==================== THAY ĐƯỜNG DẪN CỦA BẠN Ở ĐÂY ====================
BASE_DRIVE = '/content/drive/MyDrive/ISIC_YOLO'

# Đường dẫn đến folder dataset_yolo (nơi có images và labels)
DATASET_PATH = f"{BASE_DRIVE}/dataset_yolo"

# Kiểm tra xem folder có tồn tại không
print("Kiểm tra dataset:")
print(f"Train images : {len(os.listdir(f'{DATASET_PATH}/images/train')) if os.path.exists(f'{DATASET_PATH}/images/train') else 'Không tìm thấy'}")
print(f"Val images   : {len(os.listdir(f'{DATASET_PATH}/images/val')) if os.path.exists(f'{DATASET_PATH}/images/val') else 'Không tìm thấy'}")
print(f"Test images  : {len(os.listdir(f'{DATASET_PATH}/images/test')) if os.path.exists(f'{DATASET_PATH}/images/test') else 'Không tìm thấy'}")

Kiểm tra dataset:
Train images : 1815
Val images   : 389
Test images  : 390


In [4]:
data_yaml_path = f"{DATASET_PATH}/data.yaml"

yaml_content = f"""
path: {DATASET_PATH}
train: images/train
val: images/val
test: images/test

nc: 1
names: ['lesion']
"""

with open(data_yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print("✅ Đã tạo/tạo lại file data.yaml")
print(f"Đường dẫn: {data_yaml_path}")

✅ Đã tạo/tạo lại file data.yaml
Đường dẫn: /content/drive/MyDrive/ISIC_YOLO/dataset_yolo/data.yaml


In [9]:
from ultralytics import YOLO

# ==================== LOAD MODEL ====================
model = YOLO("yolo11s.pt")        # Bạn có thể đổi thành yolov8m.pt hoặc yolov11s.pt

# ==================== TRAINING CONFIG ====================
results = model.train(
    data="/content/drive/MyDrive/ISIC_YOLO/dataset_yolo/data.yaml",   # ← THAY ĐƯỜNG DẪN NÀY CHO ĐÚNG

    epochs=100,                       # Tăng từ 30 lên 100~150 vì dataset medical thường cần nhiều epoch hơn
    imgsz=640,
    batch=-1,                         # Auto batch size (tự điều chỉnh theo GPU)

    name="YOLOv11s_ISIC_Lesion",       # Tên run dễ phân biệt
    project="/content/drive/MyDrive/ISIC_YOLO/YOLO_Results",   # Lưu kết quả vào Drive

    optimizer="AdamW",
    lr0=0.01,                         # Learning rate khởi điểm cao hơn so với lá lúa
    lrf=0.01,
    patience=30,                      # Tăng patience vì train lâu hơn

    # ==================== AUGMENTATION (điều chỉnh phù hợp với ảnh da) ====================
    augment=True,
    degrees=5.0,                      # Tăng nhẹ so với lá lúa vì ảnh da có thể xoay
    translate=0.1,                    # Dịch chuyển nhiều hơn một chút
    scale=0.15,                       # Phóng to/thu nhỏ
    shear=0.0,
    flipud=0.0,                       # Không lật dọc (ảnh da thường không lật dọc)
    fliplr=0.5,                       # Lật ngang mạnh hơn (da đối xứng trái phải)
    mosaic=0.5,                       # Mosaic cao hơn so với lá lúa (hữu ích cho lesion nhỏ)
    mixup=0.1,                        # Giữ thấp để tránh làm mờ lesion
    copy_paste=0.1,                   # Thêm copy_paste cho lesion nhỏ

    # ==================== KHÁC ====================
    save=True,
    save_period=10,                   # Lưu checkpoint mỗi 10 epoch
    device=0,
    cache=True,                       # Cache dataset vào RAM (nhanh hơn)
    pretrained=True,

    warmup_epochs=3,
    warmup_momentum=0.8,
    cos_lr=True,                      # Cosine learning rate schedule (rất tốt cho medical)
    close_mosaic=10,                  # Tắt mosaic trong 10 epoch cuối (quan trọng!)

    # Box loss - Bạn có thể thử
    # box_loss="DIoU",                # Uncomment nếu muốn dùng DIoU thay vì CIoU mặc định
)

print("🎉 Training hoàn tất!")
print(f"Best model: {model.trainer.best}")

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/drive/MyDrive/ISIC_YOLO/dataset_yolo/data.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=0.5, multi_scale=0.0, name=YOLOv11s_ISIC_Lesion, nbs=64, nms=False, opset=None, optimize=False, optimizer=Ad

In [10]:
# Load best model
best_model_path = f'{BASE_DRIVE}/YOLO_Results/YOLOv11s_ISIC_Lesion/weights/best.pt'
model = YOLO(best_model_path)

print("=== ĐÁNH GIÁ TRÊN VALIDATION SET ===")
val_results = model.val(data=data_yaml_path, split='val')

print("\n=== ĐÁNH GIÁ TRÊN TEST SET ===")
test_results = model.val(data=data_yaml_path, split='test')

print("\n✅ Hoàn tất đánh giá!")

=== ĐÁNH GIÁ TRÊN VALIDATION SET ===
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 101 layers, 9,413,187 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 62.7±41.0 MB/s, size: 2071.3 KB)
val: Scanning /content/drive/MyDrive/ISIC_YOLO/dataset_yolo/labels/val.cache... 389 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 389/389 85.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 2.4s/it 1:00
                   all        389        389      0.937      0.916      0.974      0.681
Speed: 1.7ms preprocess, 8.3ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect/val

=== ĐÁNH GIÁ TRÊN TEST SET ===
Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
val: Fast image access ✅ (ping: 1.1±1.2 ms, read: 20.0±40.6 MB/s, size: 4811.8 KB)
val: Scanning /c